# 01 — Explore dataset + run the two gates

This notebook is the blocker for every downstream step. It must be run first and both gates must pass before `notebooks/02_train_qlora.ipynb` is started.

Flow:
1. Install deps, pull the project code onto Colab.
2. EDA on MACCROBAT + BioLeaflets — confirm the label vocabulary and document count.
3. **Audit Gate** — hand-audit 50 multi-drug conversions. Require ≥90% accuracy.
4. **Yield Gate** — count sentences per split; decide synthetic N + target model.
5. Generate synthetics; write `data/*.parquet`.

Output: `train.parquet`, `val.parquet`, `test.parquet`, `test_unseen_docs.parquet`.

## Setup

In [ ]:
import os, sys

def _find_project_root() -> str:
    candidates = [os.path.abspath("..")]
    candidates += [
        "/content/fineTune LLM",
        "/content/fineTune_LLM",
        "/content/finetune-llm",
        "/content/drive/MyDrive/fineTune LLM",
        "/content/drive/MyDrive/fineTune_LLM",
    ]
    if os.path.isdir("/content"):
        for entry in os.listdir("/content"):
            p = os.path.join("/content", entry)
            if os.path.isfile(os.path.join(p, "schema.py")):
                candidates.append(p)
    for c in candidates:
        if os.path.isfile(os.path.join(c, "schema.py")):
            return c
    return ""

PROJECT_ROOT = _find_project_root()
if not PROJECT_ROOT:
    raise FileNotFoundError(
        "Project files not found. On Colab, either:\n"
        "  (A) Clone your repo:\n"
        "      !git clone https://github.com/<you>/<repo>.git '/content/fineTune LLM'\n"
        "  (B) Upload via the file panel (left sidebar) to /content/fineTune LLM/\n"
        "  (C) Mount Drive and copy from MyDrive.\n"
        "Then re-run this cell."
    )
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working in:", os.getcwd())


In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import json
from collections import Counter
from pathlib import Path
import random
import pandas as pd

from data.prepare import (
    load_maccrobat, load_bioleaflets, partition_docs,
    doc_to_examples, build_all, split_into_sentences,
    group_entities_in_sentence, Entity,
)
random.seed(7)

## 1. EDA on MACCROBAT

In [ ]:
docs = load_maccrobat()
print(f"Loaded {len(docs)} documents")
ent_counts = Counter()
for d in docs:
    for e in d.entities:
        ent_counts[e.kind] += 1
print("Entity counts (normalised to our schema):")
for k, v in ent_counts.most_common():
    print(f"  {k:10s} {v:>6}")

In [ ]:
# If the entity counts look wrong (e.g. `drug` is 0), MACCROBAT's label names
# don't match `ENTITY_ALIASES` in data/prepare.py. Inspect raw label names
# and update the alias map.
from datasets import load_dataset
raw = load_dataset("singh-aditya/MACCROBAT_biomedical_ner", split="train")
print("Raw label names:")
for n in raw.features["ner_tags"].feature.names:
    print("  ", n)

## 2. Partition + sentence-level conversion on the training pool

In [ ]:
pool, test_docs, unseen_docs = partition_docs(docs)
print(f"pool={len(pool)}  test={len(test_docs)}  unseen={len(unseen_docs)}")

pool_examples = []
for d in pool:
    pool_examples.extend(doc_to_examples(d))
print(f"pool sentences (after entity grouping): {len(pool_examples)}")

## 3. **AUDIT GATE** — 50 multi-drug sentences

The proximity rule for attaching `Dosage / Route / Frequency / Duration` to the nearest `Medication` anchor is the most error-prone step in prep. We require ≥90% accuracy on 50 random multi-drug sentences before proceeding.

For each example: read the sentence, check whether every field in the JSON is attached to the correct drug. Mark 1 (correct) or 0 (incorrect). Unfilled fields are fine as long as the filled ones are right.

In [ ]:
# Sample 50 sentences with ≥2 medications
multi_drug = [
    ex for ex in pool_examples
    if len(json.loads(ex["target"]).get("medications", [])) >= 2
]
if len(multi_drug) < 50:
    print(f"[warn] only {len(multi_drug)} multi-drug sentences available; auditing all of them")
sample = random.sample(multi_drug, min(50, len(multi_drug)))
for i, ex in enumerate(sample, 1):
    print(f"--- #{i} ---")
    print("SENT:  ", ex["input"])
    print("PRED:  ", json.dumps(json.loads(ex["target"]), indent=2))
    print()

In [ ]:
# Fill this list with 50 ones and zeros in order (1=correct, 0=wrong).
# Example: scores = [1,1,0,1,1,1,1,0,1,1, ...]
scores = [1] * len(sample)  # <-- replace this placeholder before moving on

assert len(scores) == len(sample), "fill in one score per audited example"
accuracy = sum(scores) / len(scores)
print(f"Audit accuracy: {accuracy:.1%}  ({sum(scores)}/{len(scores)})")

AUDIT_THRESHOLD = 0.90
if accuracy < AUDIT_THRESHOLD:
    raise RuntimeError(
        f"Audit gate FAILED at {accuracy:.1%}; refine the grouping heuristic in "
        f"data/prepare.py (consider splitting on ';', 'and', newlines, bullet markers), "
        f"then re-run this notebook from the top."
    )
print("✓ Audit gate passed.")

## 4. **YIELD GATE** — decide synthetic count and target model

| MACCROBAT training-pool sentences | Action |
| --- | --- |
| ≥ 3,000 | Proceed: 1,500 synthetics, Qwen2.5-7B-Instruct |
| 1,500 – 3,000 | Raise synthetics to ~3,000; 7B still viable |
| < 1,500 | Downgrade to Qwen2.5-3B-Instruct; synthetics ~3,000 |

In [ ]:
n_maccrobat = len(pool_examples)
if n_maccrobat >= 3000:
    synthetic_n = 1500
    target_model = "Qwen/Qwen2.5-7B-Instruct"
elif n_maccrobat >= 1500:
    synthetic_n = 3000
    target_model = "Qwen/Qwen2.5-7B-Instruct"
else:
    synthetic_n = 3000
    target_model = "Qwen/Qwen2.5-3B-Instruct"

print(f"MACCROBAT pool sentences: {n_maccrobat}")
print(f"→ synthetic N = {synthetic_n}")
print(f"→ target model = {target_model}")

Path("artifacts").mkdir(exist_ok=True)
with open("artifacts/gate_decisions.json", "w") as f:
    json.dump({
        "audit_accuracy": accuracy,
        "maccrobat_pool_sentences": n_maccrobat,
        "synthetic_n": synthetic_n,
        "target_model": target_model,
    }, f, indent=2)
print("✓ Yield gate decision recorded at artifacts/gate_decisions.json")

## 5. Generate synthetic examples

In [ ]:
import os
if not os.environ.get("ANTHROPIC_API_KEY"):
    # on Colab, use a Secrets entry and expose it like this:
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        raise RuntimeError("set ANTHROPIC_API_KEY before running synthesis")

from data.synthesize import generate as gen_synthetic
synthetic = gen_synthetic(synthetic_n)
print(f"Generated {len(synthetic)} validated synthetic examples")
print("Sample:", synthetic[0])

## 6. Run full prep pipeline and write parquet

In [ ]:
summary = build_all(synthetic_examples=synthetic)
print(json.dumps(summary, indent=2))

# Sanity checks
for name in ("train", "val", "test", "test_unseen_docs"):
    df = pd.read_parquet(f"data/{name}.parquet")
    print(f"{name:20s}  rows={len(df):>5}  distinct doc_ids={df['doc_id'].nunique()}")

In [ ]:
# Cross-check: no synthetic in test sets; test sets disjoint by doc_id
test = pd.read_parquet("data/test.parquet")
unseen = pd.read_parquet("data/test_unseen_docs.parquet")
train = pd.read_parquet("data/train.parquet")

assert (test["source"] == "maccrobat").all(), "synthetic leaked into test.parquet"
assert (unseen["source"] == "maccrobat").all(), "synthetic leaked into test_unseen_docs.parquet"

test_ids = set(test["doc_id"])
unseen_ids = set(unseen["doc_id"])
train_ids = set(train["doc_id"])
assert test_ids.isdisjoint(unseen_ids), "test and unseen docs overlap"
assert test_ids.isdisjoint(train_ids), "test docs leaked into train"
assert unseen_ids.isdisjoint(train_ids), "unseen docs leaked into train"
print("✓ all integrity checks passed")